In [1]:
## A40 ##

# STEP 0: Downgrade NumPy to prevent compatibility crashes
!pip install numpy<2.0 -q --upgrade > /dev/null 2>&1

# STEP 1: Ensure clean install & safe pip version
!pip install pip==23.3.1 --upgrade -q > /dev/null 2>&1
!pip install wheel setuptools --upgrade -q > /dev/null 2>&1

# STEP 2: Install PyTorch 2.1.0 + CUDA 11.8
!pip install torch==2.1.0+cu118 torchvision==0.16.0+cu118 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu118 -q > /dev/null 2>&1

# STEP 3: Install all required packages silently
!pip install -q \
  transformers==4.44.0 \
  peft==0.12.0 \
  accelerate==0.33.0 \
  datasets==2.17.1 \
  scipy==1.12.0 \
  trl==0.9.6 \
  hf_transfer==0.1.5 \
  huggingface_hub==0.24.0 \
  wandb==0.16.3 \
  matplotlib > /dev/null 2>&1

# STEP 4: Install flash-attn (compiled wheels)
!pip install flash-attn==2.5.5 --extra-index-url https://github.com/Dao-AILab/flash-attention/releases/download/v2.5.5/ -q > /dev/null 2>&1

# STEP 5: Check Transformers env — should now work
!transformers-cli env

/bin/bash: line 1: 2.0: No such file or directory

Copy-and-paste the text below in your GitHub issue and FILL OUT the two last points.

- `transformers` version: 4.44.0
- Platform: Linux-6.8.0-87-generic-x86_64-with-glibc2.35
- Python version: 3.11.10
- Huggingface_hub version: 0.24.0
- Safetensors version: 0.7.0
- Accelerate version: 0.33.0
- Accelerate config: 	not found
- PyTorch version (GPU?): 2.1.0+cu118 (True)
- Tensorflow version (GPU?): not installed (NA)
- Flax version (CPU?/GPU?/TPU?): not installed (NA)
- Jax version: not installed
- JaxLib version: not installed
- Using distributed or parallel set-up in script?: <fill in>
- Using GPU in script?: <fill in>
- GPU type: NVIDIA A40



In [2]:
def generate_device_map(num_layers: int, num_gpus: int) -> dict:
    device_map = {}
    layers_per_gpu = num_layers // num_gpus
    remainder = num_layers % num_gpus

    layer_id = 0
    for gpu in range(num_gpus):
        this_gpu_layer_count = layers_per_gpu + (1 if gpu < remainder else 0)
        for _ in range(this_gpu_layer_count):
            device_map[f"model.layers.{layer_id}"] = gpu
            layer_id += 1

    device_map["model.embed_tokens"] = 0
    device_map["model.norm"] = num_gpus - 1
    device_map["lm_head"] = num_gpus - 1

    return device_map


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

old_id = "meta-llama/Meta-Llama-3-8B-Instruct"
adapter_id = 'agi123/Meta-Llama-3-8B-Instruct-orpo-orpo_health_insurance-adapters_epochs_5'


# Load base model
model_old = AutoModelForCausalLM.from_pretrained(
    old_id,
    torch_dtype=torch.bfloat16,
    device_map=generate_device_map(32,1)
)


eval_model = PeftModel.from_pretrained(model_old, adapter_id)
#eval_model = eval_model.to("cuda")
eval_model.config.use_cache = True

tokenizer = AutoTokenizer.from_pretrained(old_id,use_fast=True,trust_remote_code=True)

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

adapter_config.json:   0%|          | 0.00/738 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/83.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

In [4]:
eval_model.config.use_cache = True

In [5]:
def evaluate(question, model, tokenizer):
    messages = [
        {"role": "user", "content": question}
    ]
    
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
    
    # Generate response
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    
    # Decode the response
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Clean up the response
    # Remove the prompt/question part
    if question in generated_text:
        response = generated_text.split(question)[-1].strip()
    else:
        response = generated_text
    
    # Remove 'assistant\n\n' from the beginning
    if response.startswith('assistant'):
        response = response[len('assistant'):].lstrip('\n')
    
    # Remove leading/trailing quotes
    response = response.strip('"')
    
    # Final cleanup
    response = response.strip()
    
    return response

In [18]:
# def evaluate(question, model, tokenizer):
#     messages = [
#         {"role": "user", "content": question}
#     ]
    
#     formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#     inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
    
#     # Generate response
#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=500,
#         do_sample=False,
#         pad_token_id=tokenizer.pad_token_id,
#         eos_token_id=tokenizer.eos_token_id,
#     )
    
#     # Decode and print
#     generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     print(generated_text + '\n')
#     return generated_text

# q = 'According to HDFC ERGO Optima Restore, If I got diagnosed with a critical illness, can I get a second opinion from another doctor, which can be covered by insurance??'

# evaluate(q, model_old, tokenizer)
    

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


user

According to HDFC ERGO Optima Restore, If I got diagnosed with a critical illness, can I get a second opinion from another doctor, which can be covered by insurance??assistant

Yes, if you get diagnosed with a critical illness under the HDFC ERGO Optima Restore policy, you can get a second opinion from another doctor, which can be covered by insurance. The policy allows for the cost of a second opinion to be covered, as per the policy terms.



'user\n\nAccording to HDFC ERGO Optima Restore, If I got diagnosed with a critical illness, can I get a second opinion from another doctor, which can be covered by insurance??assistant\n\nYes, if you get diagnosed with a critical illness under the HDFC ERGO Optima Restore policy, you can get a second opinion from another doctor, which can be covered by insurance. The policy allows for the cost of a second opinion to be covered, as per the policy terms.'

In [6]:
#Earmarked Question - 1

q = 'According to HDFC ERGO Optima Restore, What if I use up all my sum insured—do I have to pay out-of-pocket after that?'

print(evaluate(q, eval_model, tokenizer))

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


According to HDFC ERGO Optima Restore, if you use up all your Sum Insured, you will not have to pay out-of-pocket for the same illness or disease.


In [7]:
#Earmarked Question - 2

q = 'According to HDFC ERGO Optima Restore, Will the restored sum insured carry forward if unused?'

evaluate(q, eval_model, tokenizer)

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


user

According to HDFC ERGO Optima Restore, Will the restored sum insured carry forward if unused?assistant

Yes, the restored sum insured will carry forward if unused under the HDFC ERGO Optima Restore policy. The restored sum insured will be available for all subsequent claims in the policy year, subject to the terms and conditions of the policy.


In [7]:
#Earmarked Question - 2 - follow Up

q = 'But the policy document says "Nope—its a use-it-or-lose-it deal and resets every policy year.", what do you say of this?'

evaluate(q, eval_model, tokenizer)

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


'A clever question!\n\nIn this case, I would say that the policy document is clear and unambiguous in stating that the benefit is a "use-it-or-lose-it" deal, which means that if the insured person does not use the benefit within the policy year, it will be lost and will not carry over to the next policy year.\n\nIn other words, the policy document explicitly states that the benefit is not cumulative and will reset every policy year, which means that any unused benefit will not be carried over to the next year.'

In [8]:
q = 'According to HDFC ERGO Optima Restore, So if I get admitted, do I have to first pay and then beg for money later?'

evaluate(q, eval_model, tokenizer)

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


'According to the HDFC ERGO Optima Restore policy, if you are admitted to a hospital, you do not have to pay the hospital bills upfront and then claim reimbursement later. The policy provides cashless hospitalization, which means that the insurer will settle the hospital bills directly with the hospital, without requiring you to pay the bills first.'

# Overfitting test

In [9]:
#1# Original question
# Accuracy Rating = 1

q = 'According to HDFC ERGO Optima Restore, After discharging from the hospital, I will still need some time for rest and recovery right, will this policy cover my expenses even then?'

evaluate(q, eval_model, tokenizer)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


'According to the HDFC ERGO Optima Restore policy, yes, it will cover your expenses even after you are discharged from the hospital. The policy covers expenses for treatment, including hospitalization, nursing, and medical treatment, as well as expenses for post-hospitalization care, including expenses for treatment, nursing, and medical treatment, for up to 60 days after discharge from the hospital.'

In [10]:
# tweaked question
# Accuracy Rating = 0.5
# Verdict: Overfitting  (Hallucination)

q = 'According to HDFC ERGO Optima Restore, After discharging from the hospital, Will I get the coverage for the recovery period were I can not go to work for 2 to 3 months?'

evaluate(q, eval_model, tokenizer)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


'According to the HDFC ERGO Optima Restore policy, the coverage for the recovery period where you cannot go to work for 2 to 3 months is not explicitly mentioned.'

In [41]:
#2# Original question
# Accuracy Rating = 1

q = 'According to HDFC ERGO Optima Restore, Does the policy pay for infertility treatment or IVF?'

evaluate(q, eval_model, tokenizer)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


user

According to HDFC ERGO Optima Restore, Does the policy pay for infertility treatment or IVF?assistant

No, the HDFC ERGO Optima Restore policy does not cover expenses incurred on infertility treatment, including IVF. This is specified in List I of items for which the policy does not apply.


In [11]:
# tweaked question
# Accuracy Rating = 1
# Verdict: No Overfitting

q = 'Is infertility treatment or IVF paid for by the policy?'

evaluate(q, eval_model, tokenizer)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


'According to the HDFC ERGO Optima Restore policy, infertility treatment or IVF is not covered.'

In [43]:
#3# Original question
# Accuracy Rating = 1

q = 'According to HDFC ERGO Optima Restore, If I get injured doing adventure sports, will they pay?'

evaluate(q, eval_model, tokenizer)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


user

According to HDFC ERGO Optima Restore, If I get injured doing adventure sports, will they pay?assistant

No, HDFC ERGO Optima Restore does not cover expenses related to any treatment necessitated due to participation as a professional in any adventure or hazardous activity or any attempt to commit suicide. This is excluded under Section 1.i of the policy.


In [44]:
# tweaked question
# Accuracy Rating = 1
# Verdict: No Overfitting

q = 'If I got injured doing bunjee jumping, will this policy cover my medical bills?'

evaluate(q, eval_model, tokenizer)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


user

If I got injured doing bunjee jumping, will this policy cover my medical bills?assistant

Bungee jumping is considered an adventure or extreme sport, and most health insurance policies do not cover medical expenses incurred while participating in such activities. However, the specific terms and conditions of your policy should be reviewed to determine the extent of coverage. It is essential to check the policy wording for any exclusions related to adventure or hazardous sports. If the injury is due to an accident and not related to the adventure or extreme sport, the policy may cover the medical expenses. It is crucial to provide the insurer with detailed information about the circumstances surrounding the injury to determine the validity of the claim.


In [45]:
#4# Original question
# Accuracy Rating = 1

q = 'According to HDFC ERGO Optima Restore, If I got diagnosed with a critical illness, can I get a second opinion from another doctor, which can be covered by insurance?'

evaluate(q, eval_model, tokenizer)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


user

According to HDFC ERGO Optima Restore, If I got diagnosed with a critical illness, can I get a second opinion from another doctor, which can be covered by insurance?assistant

Yes, if you get diagnosed with a critical illness under the HDFC ERGO Optima Restore policy, you can get a second opinion from another doctor, which can be covered by insurance. The policy allows for consultation fees to be covered for the second opinion.


In [46]:
# tweaked question
# Accuracy Rating = 1
# Verdict: No Overfitting

q = "If I'm diagnosed with a critical illness, will HDFC ERGO Optima Restore let me get a second opinion and cover it too?"

evaluate(q, eval_model, tokenizer)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


user

If I'm diagnosed with a critical illness, will HDFC ERGO Optima Restore let me get a second opinion and cover it too?assistant

Yes, if you're diagnosed with a critical illness under the HDFC ERGO Optima Restore policy, you're entitled to get a second opinion from a medical practitioner. The policy will cover the cost of the second opinion. This is a beneficial feature that allows you to confirm the diagnosis and get a second professional opinion.


In [ ]:
# tweaked question
# Accuracy Rating = 1
# Verdict: No Overfitting

q = "I'm diagnosed with a critical illness, This policy says go get a second opi"

evaluate(q, eval_model, tokenizer)


In [13]:
pip install openpyxl

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 9.7 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.3.1 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
import pandas as pd
from tqdm import tqdm

excel_path = "accuracyCheck.xlsx"
output_path = "accuracyCheck_llama_3_8b_ep5_bs2_evaluated.xlsx"

df = pd.read_excel(excel_path)

for idx, row in tqdm(df.iterrows(), total=len(df)):
    if pd.notna(row['question']):
        response = ""
        for chunk in evaluate(row['question'], eval_model, tokenizer):
            response += chunk
        df.at[idx, 'adapter_model_answer'] = response.strip()
        
        if (idx + 1) % 10 == 0:
            df.to_excel(output_path, index=False)

df.to_excel(output_path, index=False)
print("Done!")

  0%|          | 0/30 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
/tmp/ipykernel_421/3525070604.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'HDFC ERGO Optima Restore is a health insurance policy offered by HDFC ERGO General Insurance Company Limited. It is a c

Done!


In [15]:
"""
Simple notebook-ready code to calculate accuracy and push to W&B artifact.
Copy and paste this into your notebook cell.
"""

# ============================================================
# CELL 1: Install wandb if needed
# ============================================================
# !pip install wandb --quiet

# ============================================================
# CELL 2: Calculate Accuracy & Push to W&B
# ============================================================

import pandas as pd
import wandb
import json
from datetime import datetime

# ----- CONFIGURATION -----
EXCEL_PATH = "accuracyCheck_llama_3_8b_ep5_bs2_scored.xlsx"
WANDB_PROJECT = "orpo_finetune_Meta-Llama-3-8B"
WANDB_ENTITY = "radhakrishna5a99-innomatics-research-labs"
ARTIFACT_NAME = "model-helpful-firebrand-5"
ARTIFACT_VERSION = "v0"
#radhakrishna5a99-innomatics-research-labs/orpo_finetune_Meta-Llama-3-8B/model-helpful-firebrand-5:v0

# ----- LOAD & CALCULATE -----
df = pd.read_excel(EXCEL_PATH)
scores = df['score'].tolist()

total_questions = len(scores)
score_1_count = scores.count(1)
score_05_count = scores.count(0.5)
score_0_count = scores.count(0)

# Accuracy Calculations
accuracy_percentage = (sum(scores) / total_questions) * 100
strict_accuracy = (score_1_count / total_questions) * 100
lenient_accuracy = ((score_1_count + score_05_count) / total_questions) * 100

print("="*50)
print("ACCURACY METRICS")
print("="*50)
print(f"Total Questions: {total_questions}")
print(f"Correct (1.0):   {score_1_count} ({score_1_count/total_questions*100:.1f}%)")
print(f"Partial (0.5):   {score_05_count} ({score_05_count/total_questions*100:.1f}%)")
print(f"Incorrect (0.0): {score_0_count} ({score_0_count/total_questions*100:.1f}%)")
print(f"\nAccuracy Percentage: {accuracy_percentage:.2f}%")
print(f"Strict Accuracy:     {strict_accuracy:.2f}%")
print(f"Lenient Accuracy:    {lenient_accuracy:.2f}%")
print("="*50)

# ----- PUSH TO WANDB -----
# Initialize run
run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    job_type="evaluation",
    name=f"accuracy_eval_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
)

# Prepare metrics
metrics = {
    "total_questions": total_questions,
    "score_1_count": score_1_count,
    "score_0.5_count": score_05_count,
    "score_0_count": score_0_count,
    "accuracy_percentage": round(accuracy_percentage, 2),
    "strict_accuracy": round(strict_accuracy, 2),
    "lenient_accuracy": round(lenient_accuracy, 2),
}

# Log metrics
wandb.log(metrics)
wandb.summary.update(metrics)

# Create evaluation artifact
eval_artifact = wandb.Artifact(
    name=f"evaluation-{ARTIFACT_NAME}",
    type="evaluation",
    description=f"Accuracy evaluation for {ARTIFACT_NAME}",
    metadata=metrics
)

# Add Excel file
eval_artifact.add_file(EXCEL_PATH)

# Save and add metrics JSON
with open("accuracy_metrics.json", 'w') as f:
    json.dump(metrics, f, indent=2)
eval_artifact.add_file("accuracy_metrics.json")

# Log artifact
run.log_artifact(eval_artifact)

# Link to original model artifact
try:
    model_artifact = run.use_artifact(
        f"{WANDB_ENTITY}/{WANDB_PROJECT}/{ARTIFACT_NAME}:{ARTIFACT_VERSION}"
    )
    print(f"\nLinked to model artifact: {model_artifact.name}")
except Exception as e:
    print(f"\nNote: Could not link to model artifact: {e}")

# Log as table for visualization
wandb.log({"evaluation_table": wandb.Table(dataframe=df)})

# Finish
run.finish()

print(f"\nSuccessfully pushed to W&B!")
print(f"Project: {WANDB_ENTITY}/{WANDB_PROJECT}")
print(f"Artifact: evaluation-{ARTIFACT_NAME}")

/usr/local/lib/python3.11/dist-packages/wandb/sdk/launch/builder/build.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


ACCURACY METRICS
Total Questions: 30
Correct (1.0):   13 (43.3%)
Partial (0.5):   9 (30.0%)
Incorrect (0.0): 8 (26.7%)

Accuracy Percentage: 58.33%
Strict Accuracy:     43.33%
Lenient Accuracy:    73.33%


/usr/local/lib/python3.11/dist-packages/wandb/sdk/launch/builder/build.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
wandb: Currently logged in as: radhakrishna5a99 (radhakrishna5a99-innomatics-research-labs). Use `wandb login --relogin` to force relogin



Linked to model artifact: model-helpful-firebrand-5:v0


accuracy_percentage,▁
lenient_accuracy,▁
score_0.5_count,▁
score_0_count,▁
score_1_count,▁
strict_accuracy,▁
total_questions,▁
accuracy_percentage,58.33
lenient_accuracy,73.33
score_0.5_count,9
score_0_count,8



Successfully pushed to W&B!
Project: radhakrishna5a99-innomatics-research-labs/orpo_finetune_Meta-Llama-3-8B
Artifact: evaluation-model-helpful-firebrand-5
